In [5]:
import pandas as pd
import numpy as np
from understatapi import UnderstatClient

In [6]:
TEAM_NAME = "Tottenham"
SEASONS = ["2022", "2023", "2024", "2025"]

In [7]:
match_states = []

with UnderstatClient() as understat:
    for season in SEASONS:
        try:
            context_data = understat.team(team=TEAM_NAME).get_context_data(season=season)
            match_state_dict = context_data.get("gameState", {})

            for goal_diff, stats in match_state_dict.items():
                row = {
                    "season": season,
                    "goal_diff": goal_diff,
                    "time": stats.get("time", 0),
                    "shots": stats.get("shots", 0),
                    "goals": stats.get("goals", 0),
                    "xG": stats.get("xG", 0),
                    "shots_against": stats.get("against", {}).get("shots", 0),
                    "goals_against": stats.get("against", {}).get("goals", 0),
                    "xGA": stats.get("against", {}).get("xG", 0)
                }
                match_states.append(row)
        except Exception as e:
            print(f"Error fetching data for season {season}: {e}")

df_raw = pd.DataFrame(match_states)

In [11]:
df = df_raw.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   season         20 non-null     str    
 1   goal_diff      20 non-null     str    
 2   time           20 non-null     int64  
 3   shots          20 non-null     int64  
 4   goals          20 non-null     int64  
 5   xG             20 non-null     float64
 6   shots_against  20 non-null     int64  
 7   goals_against  20 non-null     int64  
 8   xGA            20 non-null     float64
dtypes: float64(2), int64(5), str(2)
memory usage: 1.5 KB


In [15]:
df.tail()

,season,goal_diff,time,shots,goals,xG,shots_against,goals_against,xGA
15,2025,Goal diff 0,1766,189,20,20.368882,206,29,25.097431
16,2025,Goal diff -1,848,114,12,12.007545,124,12,16.254450
17,2025,Goal diff +1,449,46,6,6.156299,66,10,6.495562
18,2025,Goal diff < -1,343,52,7,5.915664,43,5,5.390593
19,2025,Goal diff > +1,273,35,3,4.486533,26,1,2.888300


In [18]:
df["xG_diff"] = df["goals"] - df["xG"]
df["xGA_diff"] = df["goals_against"] - df["xGA"]

# シーズンの総得点数と時間帯別の割合を算出
df["season_total_goals"] = df.groupby("season")["goals"].transform("sum")
df["goal_percentage"] = (df["goals"] / df["season_total_goals"]) * 100
# シーズンの総失点数と時間帯別の割合を算出
df["season_total_conceded"] = df.groupby("season")["goals_against"].transform("sum")
df["conceded_percentage"] = (df["goals_against"] / df["season_total_conceded"]) * 100

In [20]:
df

,season,goal_diff,time,shots,goals,xG,shots_against,goals_against,xGA,xG_diff,xGA_diff,season_total_goals,season_total_conceded,goal_percentage,conceded_percentage
0,2022,Goal diff 0,1541,215,28,23.624071,207,27,20.919183,4.375929,6.080817,70,63,40.000000,42.857143
1,2022,Goal diff +1,822,108,13,14.060679,119,10,8.152824,-1.060679,1.847176,70,63,18.571429,15.873016
2,2022,Goal diff -1,632,97,13,12.398329,79,15,12.411409,0.601671,2.588591,70,63,18.571429,23.809524
3,2022,Goal diff < -1,395,66,9,6.234513,64,6,7.140909,2.765487,-1.140909,70,63,12.857143,9.523810
4,2022,Goal diff > +1,226,36,7,4.119477,51,5,4.402077,2.880523,0.597923,70,63,10.000000,7.936508
5,2023,Goal diff 0,1678,246,31,31.562179,197,27,32.507935,-0.562179,-5.507935,74,61,41.891892,44.262295
6,2023,Goal diff +1,778,99,14,14.665255,115,11,15.130763,-0.665255,-4.130763,74,61,18.918919,18.032787
7,2023,Goal diff -1,523,108,15,15.216742,62,8,9.818866,-0.216742,-1.818866,74,61,20.270270,13.114754
8,2023,Goal diff > +1,414,72,8,9.635498,69,6,6.465000,-1.635498,-0.465000,74,61,10.810811,9.836066
9,2023,Goal diff < -1,309,62,6,7.677822,47,9,8.274268,-1.677822,0.725732,74,61,8.108108,14.754098
